# Sprint 5B — Graph C Energy Sensitivity Runner

**Runner-only notebook. Model, preprocessing, evaluation ve plot logic notebook içinde değildir.**
Tüm bilimsel kod `src/`, `scripts/`, `configs/` altındadır.

Execution plan: `docs/exec-plans/active/005-sprint5-epigenetic-ablation.md` Slice 6  
Runner boundary: `colab/README.md`

---
**Başlamadan önce kontrol et:**
- [ ] Colab runtime GPU seçildi mi?
- [ ] Drive'da `crispr_gnn_offtarget/data/` altında raw dataset veya processed parquet var mı?
- [ ] Branch GitHub'a push edildi mi? (`sprint5/epigenetic-ablation`)
- [ ] Bu run secondary sensitivity olarak raporlanacak, primary feature ablation olarak yorumlanmayacak.


## ADIM 1 — Google Drive Mount


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## ADIM 2 — Repo Clone ve Checkout


In [ ]:
%%bash
set -euo pipefail
pip install uv --quiet
git clone https://github.com/YasinEkici/crispr-gnn-offtarget.git crispr-gnn-offtarget
cd crispr-gnn-offtarget
git checkout sprint5/epigenetic-ablation
echo "=== Commit SHA ==="
git rev-parse HEAD


## ADIM 3 — Dependency Sync ve GPU Kontrolü


In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
uv sync
uv run python - <<'PY'
import torch
try:
    import torch_geometric
    pyg_version = torch_geometric.__version__
except Exception as exc:
    pyg_version = f'unavailable: {exc}'
print('torch         :', torch.__version__)
print('pyg           :', pyg_version)
print('cuda_available:', torch.cuda.is_available())
print('cuda_version  :', torch.version.cuda)
print('device        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
PY


## ADIM 4 — Drive Data Kopyala

Bu cell Drive'daki proje data klasörünü Colab lokal diske kopyalar. Büyük dosyalar repo'ya commit edilmez.


In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
DRIVE_ROOTS=("/content/drive/MyDrive/crispr_gnn_offtarget" "/content/drive/MyDrive/crispr-gnn-offtarget")
mkdir -p data
for DRIVE_ROOT in "${DRIVE_ROOTS[@]}"; do
  if [ -d "$DRIVE_ROOT/data/raw" ]; then
    mkdir -p data/raw
    rsync -a "$DRIVE_ROOT/data/raw/" data/raw/
    echo "Copied raw from $DRIVE_ROOT"
  fi
  if [ -d "$DRIVE_ROOT/data/processed" ]; then
    mkdir -p data/processed
    rsync -a "$DRIVE_ROOT/data/processed/" data/processed/
    echo "Copied processed from $DRIVE_ROOT"
  fi
done
if [ ! -f data/processed/graphs/sprint3/graph_c_context_observation/manifest.json ]; then
  echo "Missing required Sprint 3 Graph C artifact: data/processed/graphs/sprint3/graph_c_context_observation/manifest.json" >&2
  echo "Check that Drive has /content/drive/MyDrive/crispr_gnn_offtarget/data/processed/graphs/sprint3" >&2
  exit 1
fi
if [ ! -f data/processed/graphs/sprint5/graph_a_minimal_physical_target/features_S5F2_energy.parquet ] && [ ! -f data/raw/260520_putative_nucleosomal.parquet ]; then
  echo "Missing S5F2 source. Need either Sprint 5 feature table or raw Mak parquet." >&2
  echo "Expected one of:" >&2
  echo "  data/processed/graphs/sprint5/graph_a_minimal_physical_target/features_S5F2_energy.parquet" >&2
  echo "  data/raw/260520_putative_nucleosomal.parquet" >&2
  exit 1
fi
find data -maxdepth 4 -type f | sort | head -50


## ADIM 5 — Sprint 5B Graph C Artefact Üret


In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
uv run python scripts/build_sprint5b_graph_c_energy_features.py \
  --data-config configs/data/mak2022.yaml \
  --schema-config configs/sweeps/graph_schema_ablation.yaml \
  --source-artifact-dir data/processed/graphs/sprint3 \
  --artifact-dir data/processed/graphs/sprint5b \
  --report-path outputs/sprint5b/graph_c_energy_sensitivity_artifact_report.md
PYTHONPATH=src uv run python - <<'PY'
from pathlib import Path
from crispr_gnn.graph.graph_schemas import GRAPH_C
from crispr_gnn.graph.pyg_dataset import Sprint3HeteroDataLoader
g = Sprint3HeteroDataLoader(Path('data/processed/graphs/sprint5b')).load(GRAPH_C)
print(g.manifest['feature_tables'])
PY


## ADIM 6 — Sprint 5B Graph C Train

Bu cell tek secondary sensitivity run başlatır. Test sonucuna göre feature/hyperparameter değiştirme yapılmaz.


In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
uv run python scripts/train.py --config configs/sweeps/sprint5b_graph_c_energy_sensitivity.yaml
RUN_DIR=$(find outputs/sprint5b/graph_c -mindepth 1 -maxdepth 1 -type d | sort | tail -1)
basename "$RUN_DIR" > /content/sprint5b_run_id.txt
cat /content/sprint5b_run_id.txt


## ADIM 7 — Returned Outputs Drive'a Kopyala


In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
DRIVE_ROOTS=("/content/drive/MyDrive/crispr_gnn_offtarget" "/content/drive/MyDrive/crispr-gnn-offtarget")
DRIVE_ROOT=""
for CANDIDATE in "${DRIVE_ROOTS[@]}"; do
  if [ -d "$CANDIDATE" ]; then
    DRIVE_ROOT="$CANDIDATE"
    break
  fi
done
if [ -z "$DRIVE_ROOT" ]; then
  echo "No Drive project root found under MyDrive" >&2
  exit 1
fi
RUN_BASENAME=$(cat /content/sprint5b_run_id.txt)
SCHEMA_OUT="outputs/sprint5b/graph_c"
DRIVE_OUT="$DRIVE_ROOT/returned_outputs/$RUN_BASENAME"
mkdir -p "$DRIVE_ROOT/returned_outputs"
if [ -e "$DRIVE_OUT" ]; then
  echo "Output already exists in Drive: $DRIVE_OUT" >&2
  exit 1
fi
mkdir -p "$DRIVE_OUT"
cp -r "$SCHEMA_OUT/$RUN_BASENAME" "$DRIVE_OUT/$RUN_BASENAME"
cp "$SCHEMA_OUT/gcn_graph_c_results.csv" "$DRIVE_OUT/gcn_graph_c_results.csv"
cp "$SCHEMA_OUT/gcn_graph_c_report.md" "$DRIVE_OUT/gcn_graph_c_report.md"
cp -r "$SCHEMA_OUT/diagnostics" "$DRIVE_OUT/diagnostics"
cp -r "$SCHEMA_OUT/figures" "$DRIVE_OUT/figures"
cp "outputs/sprint5b/graph_c_energy_sensitivity_artifact_report.md" "$DRIVE_OUT/graph_c_energy_sensitivity_artifact_report.md"
echo "Copied to: $DRIVE_OUT"
find "$DRIVE_OUT" -maxdepth 3 -type f | sort | head -100


## ADIM 8 — Kısa Sonuç Özeti


In [ ]:
%%bash
set -euo pipefail
cd crispr-gnn-offtarget
RESULTS="outputs/sprint5b/graph_c/gcn_graph_c_results.csv"
uv run python - <<PY
import pandas as pd
results = pd.read_csv('$RESULTS')
cols = ['feature_set','test_auprc','test_auroc','test_f1','test_macro_f1','test_mcc','test_specificity','test_tn','test_fp','test_fn','test_tp']
print(results[cols].to_string(index=False))
PY
